# Meshing and simulating the MaMMoS word mark

This notebook creates a three-dimensional **MaMMoS** word mark with PyVista, converts its closed surface into a tetrahedral volume mesh with TetGen, prepares a single-phase micromagnetic model, and runs a minimal four-point hysteresis calculation.

In [ ]:
import warnings

from matplotlib.colors import LinearSegmentedColormap
import micromagneticmodel as mm
import numpy as np
import pyvista as pv
import tetgen

import tommos_adapter as ta

pv.set_jupyter_backend("trame")

## Create a closed MaMMoS surface

PyVista creates the word mark in the $xy$ plane and extrudes it along $z$. The coordinates below are mesh units representing nanometres. A height of 40 and a depth of 10 therefore correspond to $40\,\mathrm{nm}$ and $10\,\mathrm{nm}$. Setting the centre explicitly places the geometry around the origin, as required by the later tommos workflow.

In [ ]:
surface = (
    pv.Text3D(
        "MaMMoS",
        height=40.0,
        depth=10.0,
        center=(0, 0, 0),
    )
    .triangulate()
    .clean()
)

if surface.n_open_edges != 0:
    raise RuntimeError(f"The word-mark surface has {surface.n_open_edges} open edges.")

surface

## Generate and inspect the tetrahedral mesh

TetGen fills the closed surface with linear tetrahedra. `max_tetrahedron_volume` is expressed in cubic mesh units, here $\mathrm{nm}^3$: smaller values enforce a finer mesh, whereas larger values allow a coarser mesh. Surface and quality constraints impose a lower bound on the resulting cell count. `fixedvolume=True` activates the volume constraint, and the radius-edge ratio is kept at a moderate value so that this first word-mark simulation remains compact.

In [ ]:
max_tetrahedron_volume = 50.0  # nm^3; decrease this value for a finer mesh

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="`PolyData._connectivity_array` is deprecated.*",
        category=pv.PyVistaDeprecationWarning,
    )
    tet = tetgen.TetGen(surface)
    tet.tetrahedralize(
        order=1,
        minratio=3.0,
        fixedvolume=True,
        maxvolume=max_tetrahedron_volume,
    )
mesh = tet.grid

if not np.all(mesh.celltypes == pv.CellType.TETRA):
    raise RuntimeError("TetGen generated non-tetrahedral cells.")

np.testing.assert_allclose(mesh.center, (0, 0, 0), atol=1e-12)
print(f"Mesh points: {mesh.n_points}")
print(f"Tetrahedra: {mesh.n_cells}")
mesh

In [ ]:
mesh_plotter = pv.Plotter()
mesh_plotter.add_mesh(mesh, show_edges=True, color="lightsteelblue")
mesh_plotter.view_vector((0.1, -0.1, 1.0), viewup=(0, 1, 0))
mesh_plotter.show(jupyter_backend="trame")

## Define a single-phase micromagnetic model

The visualization mesh remains in nanometres. A deep copy is converted to SI units for the micromagnetic model. The adapter expects one material identifier and one saturation-magnetization value per tetrahedron, while the initial unit magnetization is defined at mesh points.

In [ ]:
Ms = 8e5
A = 7e-12  # exchange stiffness in J/m
K = 1e4  # uniaxial anisotropy constant in J/m^3
u = (0, 0, 1)  # easy-axis direction

In [ ]:
mesh_si = mesh.copy(deep=True)
mesh_si.points *= ta.scripts.DEFAULT_MESH_UNIT

mesh_si.cell_data["region_id"] = np.ones(mesh_si.n_cells, dtype=np.int32)
mesh_si.cell_data["Ms"] = np.full(mesh_si.n_cells, 8e5)
mesh_si.point_data["m"] = np.tile((0.0, 0.0, 1.0), (mesh_si.n_points, 1))

mesh_si

The model contains exchange, uniaxial anisotropy with its easy axis along $z$, and demagnetization energy. These material parameters are illustrative and can be refined before running a hysteresis calculation.

In [ ]:
system = mm.System(name="mammos_wordmark", m=mesh_si)
system.energy = mm.Exchange(A=A) + mm.UniaxialAnisotropy(K=K, u=u) + mm.Demag()
system.T = 0

system

## Run a minimal hysteresis calculation

The applied field follows the uniaxial easy axis and sweeps from $+2\,\mathrm{T}$ to $-2\,\mathrm{T}$. Four field points keep this run short; they demonstrate the complete workflow but are not sufficient for a physically resolved hysteresis curve.

In [ ]:
field_limit = 2.0 / mm.consts.mu0
driver = ta.HysteresisDriver()
driver.drive(
    system,
    h_direction=(0, 0.001, 1),
    h_start=field_limit,
    h_final=0.0,
    n=4,
)

## Plot the final magnetization without the Airshell

After the drive, the adapter maps the final nodal magnetization back onto the original PyVista mesh. Consequently, `system.m` contains only the magnetic MaMMoS geometry and not the auxiliary tommos Airshell. The glyphs below are centered on their mesh points and show the final magnetization vectors, colored by their $y$ component.

In [ ]:
# NBVAL_SKIP
magnetic_state = system.m.extract_cells(np.asarray(system.m.cell_data["Ms"]) > 0)
magnetic_state.point_data["m_x"] = np.asarray(magnetic_state.point_data["m"])[:, 0]
magnetic_state.point_data["m_y"] = np.asarray(magnetic_state.point_data["m"])[:, 1]
magnetic_state.point_data["m_z"] = np.asarray(magnetic_state.point_data["m"])[:, 2]

domain_lengths = np.array(
    [
        magnetic_state.bounds[1] - magnetic_state.bounds[0],
        magnetic_state.bounds[3] - magnetic_state.bounds[2],
        magnetic_state.bounds[5] - magnetic_state.bounds[4],
    ]
)
glyph_length = 0.05 * domain_lengths.max()

centered_arrow = pv.Arrow(start=(-0.5, 0, 0), direction=(1, 0, 0))
magnetization_glyphs = magnetic_state.glyph(
    orient="m",
    scale=False,
    factor=glyph_length,
    geom=centered_arrow,
    tolerance=None,
)

mammos_cmap = LinearSegmentedColormap.from_list(
    "mammos", ["#62B7CF", "#F7F7F7", "#2CB877"]
)

vector_plotter = pv.Plotter()
vector_plotter.add_mesh(
    magnetic_state.extract_surface(algorithm="dataset_surface"),
    scalars="region_id",
    categories=True,
    opacity=0.15,
    show_scalar_bar=False,
)
vector_plotter.add_mesh(
    magnetization_glyphs,
    scalars="m_y",
    cmap=mammos_cmap,
    clim=(-1, 1),
    scalar_bar_args={"title": "m_y"},
)
vector_plotter.add_axes()
# Show the xy plane almost face-on while retaining a slight three-dimensional tilt.
vector_plotter.view_vector((0.1, -0.1, 1.0), viewup=(0, 1, 0))
vector_plotter.show(jupyter_backend="trame")